# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [1]:
# Optional setup: install dependencies if they are missing in your environment.
%pip install -q transformers torch


In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "hello, my name's cyriac, are you ok?"
print(sample_sentence)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

hello, my name's cyriac, are you ok?


In [3]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | hello        |  7592
    2 | ,            |  1010
    3 | my           |  2026
    4 | name         |  2171
    5 | '            |  1005
    6 | s            |  1055
    7 | cy           | 22330
    8 | ##ria        |  4360
    9 | ##c          |  2278
   10 | ,            |  1010
   11 | are          |  2024
   12 | you          |  2017
   13 | ok           |  7929
   14 | ?            |  1029
   15 | [SEP]        |   102
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (15, '[SEP]'), (16, '[PAD]'), (17, '[PAD]'), (18, '[PAD]'), (19, '[PAD]'), (20, '[PAD]'), (21, '[PAD]'), (22, '[PAD]

### Exercise 1 reflection
- TODO: Describe how [CLS] and [SEP] behave inside the encoder.
- TODO: Explain how the attention mask hides padded positions from self-attention.


### Exercise 1 reflection

- **How [CLS] and [SEP] behave inside the encoder:**
  * Le token [CLS] est ajouté au début de la séquence d'entrée. Son état caché final correspondant issu du modèle BERT est souvent utilisé comme représentation agrégée de la séquence d'entrée entière pour les tâches de classification. Par exemple, en analyse des sentiments, ce vecteur est utilisé comme entrée d'un classificateur linéaire pour déterminer le sentiment.
  * Le token [SEP] sert à marquer la fin d'une phrase ou à séparer deux phrases lorsqu'elles sont fournies par paire à BERT. Il aide le modèle à comprendre les limites des phrases, ce qui est crucial pour des tâches telles que la réponse aux questions ou l'inférence en langage naturel, où les relations entre les phrases sont importantes.

- **How the attention mask hides padded positions from self-attention:**
  * Le masque d'attention est un tenseur binaire indiquant quels Tokens doivent être pris en compte et lesquels doivent être ignorés par le mécanisme d'auto-attention de l'encodeur BERT. Une valeur de 1 dans le masque d'attention signifie que le token correspondant est un jeton réel et doit être pris en compte, tandis qu'une valeur de 0 signifie que le jeton est un jeton de remplissage et doit être ignoré.
  * Lors du calcul de l'auto-attention, les scores d'attention des tokens de remplissage sont généralement masqués avant l'application de la fonction softmax. Ceci garantit que le modèle ne prête aucune attention aux tokens de remplissage, les empêchant ainsi d'influencer les représentations contextuelles des tokens d'entrée réels. Cela indique au modèle de se concentrer uniquement sur les parties significatives de la séquence d'entrée.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [6]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "i like"
prediction = sentiment_pipeline(sentence)
prediction


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998149275779724}]

### Exercise 2 reflection
- TODO: Does the predicted label match your expectation? Why or why not?
-> Le resultat attendu correspond à ce que je recherche, parce qu'ici j'envoi une phrase positive avec 'like'
- TODO: How confident is the model and what does the score tell you?
-> Le modèle fiable car il arrive à trouver le sens de ma phrase avec un score de 99%


## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        self.max_length = max_length

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            'input_ids': encoding['input_ids'].to(self.device),
            'attention_mask': encoding['attention_mask'].to(self.device)
        }

    def predict(self, text: str) -> Dict[str, float]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            outputs = self.model(**inputs)

        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1).squeeze().tolist()

        predicted_class_id = logits.argmax().item()
        label = self.model.config.id2label[predicted_class_id]
        score = probabilities[predicted_class_id]

        return {"label": label, "score": score}

In [8]:
# TODO: instantiate your analyzer and test several sentences once the class is ready.
analyzer = BERTSentimentAnalyzer()
samples = [
    "I like your dresscode",
    "This cours is not good"
]
for text in samples:
    print(text)
    print(analyzer.predict(text))


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

I like your dresscode
{'label': 'POSITIVE', 'score': 0.9990801811218262}
This cours is not good
{'label': 'NEGATIVE', 'score': 0.9997866749763489}


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [9]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
from typing import List, Dict

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)

    def recognize(self, text: str) -> List[Dict]:
        # Tokenize the input text, retaining offset mapping and word IDs
        encoding = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            return_offsets_mapping=True,
            return_word_ids=True
        )

        input_ids = encoding["input_ids"].to(self.device)
        attention_mask = encoding["attention_mask"].to(self.device)
        offset_mapping = encoding["offset_mapping"].squeeze().tolist()
        word_ids = encoding.word_ids(batch_index=0)

        # Run model inference
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=2).squeeze().tolist()

        # Map predicted IDs to labels (e.g., 'B-PER', 'I-LOC', 'O')
        predicted_labels = [self.model.config.id2label[p_id] for p_id in predictions]

        # --- Process token-level predictions to extract and merge entities ---
        words_info = []
        current_word_tokens_encoded = []
        current_word_labels = []
        current_word_start_char = -1
        current_word_end_char = -1
        prev_word_idx = None

        for i, (label, word_idx, offset) in enumerate(zip(predicted_labels, word_ids, offset_mapping)):
            if word_idx is None:  # Skip special tokens ([CLS], [SEP], [PAD])
                continue

            token_start_char, token_end_char = offset

            if prev_word_idx != word_idx:  # Start of a new original word
                if current_word_tokens_encoded:  # Store info for the previous word
                    words_info.append({
                        "text": text[current_word_start_char:current_word_end_char],
                        "labels": current_word_labels,  # All labels for its sub-tokens
                        "start": current_word_start_char,
                        "end": current_word_end_char
                    })
                # Reset for the new word
                current_word_tokens_encoded = [input_ids.squeeze()[i]]
                current_word_labels = [label]
                current_word_start_char = token_start_char
                current_word_end_char = token_end_char
            else:  # Continuation of the same original word (subword token)
                current_word_tokens_encoded.append(input_ids.squeeze()[i])
                current_word_labels.append(label)
                current_word_end_char = token_end_char  # Extend end to cover all subwords

            prev_word_idx = word_idx

        if current_word_tokens_encoded:  # Add the last word's info after the loop
            words_info.append({
                "text": text[current_word_start_char:current_word_end_char],
                "labels": current_word_labels,
                "start": current_word_start_char,
                "end": current_word_end_char
            })

        # Now, process word-level entities for merging into final entities
        entities = []
        current_entity_type = None
        current_entity_start = -1
        current_entity_end = -1

        for word_info in words_info:
            # Determine the effective label for the current word
            # Prioritize B- tags, then I- tags if no B- was found, else O
            word_label = 'O'
            for label in word_info['labels']:
                if label.startswith('B-'):
                    word_label = label
                    break
                elif label.startswith('I-') and word_label == 'O':
                    word_label = label

            # Extract the actual entity type (e.g., 'PER' from 'B-PER')
            entity_tag = word_label[2:] if len(word_label) > 2 and (word_label.startswith('B-') or word_label.startswith('I-')) else word_label

            if word_label.startswith('B-') or (word_label.startswith('I-') and entity_tag != current_entity_type): # Start of a new entity or new type
                if current_entity_type is not None and current_entity_type != 'O': # Finalize previous entity if any
                    entities.append({
                        "text": text[current_entity_start:current_entity_end],
                        "entity": current_entity_type,
                        "start": current_entity_start,
                        "end": current_entity_end
                    })
                current_entity_type = entity_tag if entity_tag != 'O' else None # Don't start an entity with 'O'
                current_entity_start = word_info['start']
                current_entity_end = word_info['end']
            elif word_label.startswith('I-') and entity_tag == current_entity_type: # Continuation of the current entity
                current_entity_end = word_info['end']
            else: # 'O' tag for the word, or non-contiguous 'I-'
                if current_entity_type is not None and current_entity_type != 'O': # Finalize previous entity if any
                    entities.append({
                        "text": text[current_entity_start:current_entity_end],
                        "entity": current_entity_type,
                        "start": current_entity_start,
                        "end": current_entity_end
                    })
                current_entity_type = None # Reset
                current_entity_start = -1
                current_entity_end = -1

        # After the loop, check if there's an ongoing entity to append
        if current_entity_type is not None and current_entity_type != 'O':
            entities.append({
                "text": text[current_entity_start:current_entity_end],
                "entity": current_entity_type,
                "start": current_entity_start,
                "end": current_entity_end
            })

        return entities

In [12]:
# TODO: instantiate the recognizer and test it on text that includes people, places, or organizations.
ner = BERTNamedEntityRecognizer()
sample_text = "Cyriac back from a long trip, this year he started bowling"
ner.recognize(sample_text)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'text': 'Cyriac', 'entity': 'PER', 'start': 0, 'end': 6}]

## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Transformateur encodeur uniquement avec auto-attention bidirectionnelle | Transformateur décodeur uniquement avec auto-attention causale (de gauche à droite)|
| Primary purpose | Compréhension et encodage du contexte textuel | Génération autorégressive de texte |
| Typical use cases | Classification, reconnaissance d'entités nommées (NER), analyse des sentiments | Génération de texte, chatbots, résumé, génération de code |
| Strengths | Capture le contexte à gauche et à droite d'un token | Génère un texte fluide et cohérent |
| Weaknesses | Non conçu pour la génération de texte libre | Impossible d'utiliser directement les jetons futurs lors du traitement d'une séquence |


## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1. TODO: Describe how BERT encodes queries and documents.  
-> BERT convertit les requêtes utilisateur et les documents en représentations vectorielles denses appelées plongements lexicaux. Lors de l'encodage, BERT analyse le contexte de chaque mot grâce à une auto-attention bidirectionnelle, ce qui lui permet de saisir le sens sémantique de l'ensemble du texte. Les requêtes et documents similaires tendent à produire des plongements lexicaux proches les uns des autres dans l'espace vectoriel.
2. TODO: Explain how those embeddings are stored and searched in a vector database.  
-> Les représentations vectorielles générées par BERT sont stockées dans une base de données vectorielles, Pinecone ou Chroma. Lorsqu'un utilisateur soumet une requête, BERT l'encode en un vecteur, et la base de données effectue une recherche de similarité afin de trouver les représentations vectorielles des documents les plus proches de celle de la requête. Ces voisins les plus proches sont ensuite renvoyés comme passages pertinents.
3. TODO: Outline how the retrieved passages are handed to a generative model like GPT.  
-> Après la récupération, les passages les plus pertinents sont combinés à la requête de l'utilisateur et intégrés à l'invite envoyée à un modèle génératif tel que GPT. Au lieu de se fier uniquement à ses connaissances internes, le modèle utilise le contexte récupéré pour générer une réponse plus précise et actualisée. Cette étape de récupération réduit les erreurs d'interprétation et renforce la pertinence factuelle de la réponse.
4. TODO: Provide a concrete application example (industry or product) where RAG with BERT makes sense.  
-> Un chatbot de support client est une application RAG courante. Les manuels d'entreprise, les FAQ et la documentation technique sont encodés par BERT et stockés dans une base de données vectorielle. Lorsqu'un client pose une question, le système récupère les documents les plus pertinents et les transmet à GPT, qui génère une réponse en langage naturel à partir des informations extraites. Cette approche est largement utilisée dans les assistants de connaissances d'entreprise, les systèmes d'assistance bancaire et les portails d'information du secteur de la santé.
